In [ ]:
import os
import time
import pickle
import numpy as np
import pandas as pd
import tensorflow as tf
import keras.backend as K
from tensorflow.keras.models import Model
from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras.layers import (
    Dense, Dropout, GlobalAveragePooling2D, DepthwiseConv2D, Input
)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import f1_score as sklearn_f1

print('TF version:', tf.__version__)

In [ ]:
# === Configuração ===
DISPOSITIVOS = ['dish washer', 'kettle', 'microwave', 'washing machine', 'fridge']
IMAGENS      = ['rp', 'gadf', 'gasf', 'mtf', 'tbg']
BATCH        = 32
FOLDER_I     = 'pickle_data'   # mesmos dados do notebook 1
N_RUNS       = 3               # runs por combinação

# Variantes: nome -> quantos blocos DepthwiseConv2D substituir por Fourier
VARIANTS = {
    'baseline':        0,
    'fourier_1block':  1,
    'fourier_3blocks': 3,
}

## Camada SpectralDepthwise

Substitui o `DepthwiseConv2D` por FFT 2-D por canal (sem parâmetros aprendíveis na etapa espacial).  
O stride original é preservado via `AveragePooling2D` quando necessário.

In [ ]:
class SpectralDepthwise(tf.keras.layers.Layer):
    """
    Substitui DepthwiseConv2D: FFT 2-D por canal (operação espacial sem pesos)
    seguida de AveragePooling quando strides > 1.
    
    Entrada : (B, H, W, C)
    Saída   : (B, H/s, W/s, C)  onde s = stride original
    """
    def __init__(self, strides=(1, 1), **kwargs):
        super().__init__(**kwargs)
        # normaliza para tupla de ints
        self.strides = (int(strides[0]), int(strides[1]))

    def build(self, input_shape):
        s = self.strides
        if s[0] > 1 or s[1] > 1:
            self._pool = tf.keras.layers.AveragePooling2D(
                pool_size=s, strides=s, padding='same'
            )
        else:
            self._pool = None
        super().build(input_shape)

    def call(self, x, training=None):
        # FFT 2-D opera nas 2 últimas dims → transpomos para (B, C, H, W)
        x_t  = tf.transpose(tf.cast(x, tf.complex64), [0, 3, 1, 2])
        mag  = tf.cast(tf.abs(tf.signal.fft2d(x_t)), tf.float32)
        out  = tf.transpose(mag, [0, 2, 3, 1])  # volta para (B, H, W, C)
        return self._pool(out) if self._pool is not None else out

    def get_config(self):
        return {**super().get_config(), 'strides': self.strides}

## Construtores de modelo

Usa `tf.keras.models.clone_model(clone_function=...)` para substituir os primeiros N `DepthwiseConv2D`  
por `SpectralDepthwise`, mantendo todos os demais pesos do ImageNet congelados.

In [ ]:
def build_feature_extractor(input_shape, n_fourier=0, name=None):
    """
    Constrói extrator de features baseado em MobileNetV3Large.
    
    n_fourier=0 → MobileNetV3Large original congelado
    n_fourier=N → primeiros N blocos DepthwiseConv2D → SpectralDepthwise
    
    Retorna: Model(input_shape → feature_vec)
    """
    base = MobileNetV3Large(
        input_shape=input_shape, weights='imagenet', include_top=False
    )

    if n_fourier == 0:
        for layer in base.layers:
            layer.trainable = False
        out = GlobalAveragePooling2D()(base.output)
        return Model(inputs=base.input, outputs=out,
                     name=name or 'FE_baseline')

    counter = [0]

    def clone_fn(layer):
        if isinstance(layer, DepthwiseConv2D):
            counter[0] += 1
            if counter[0] <= n_fourier:
                cfg = layer.get_config()
                strides = cfg.get('strides', (1, 1))
                return SpectralDepthwise(
                    strides=strides, name=f'sdw_{counter[0]}'
                )
        # retorna a mesma instância → pesos ImageNet compartilhados
        return layer

    cloned = tf.keras.models.clone_model(base, clone_function=clone_fn)

    for layer in cloned.layers:
        if not isinstance(layer, SpectralDepthwise):
            layer.trainable = False

    out = GlobalAveragePooling2D()(cloned.output)
    return Model(inputs=cloned.input, outputs=out,
                 name=name or f'FE_fourier_{n_fourier}')


def build_full_model(input_shape, n_fourier=0):
    """Feature extractor + cabeça MLP idêntica ao notebook 1."""
    fe = build_feature_extractor(input_shape, n_fourier)
    x  = Dense(64, activation='relu')(fe.output)
    x  = Dropout(0.25)(x)
    x  = Dense(64, activation='relu')(x)
    x  = Dropout(0.25)(x)
    out = Dense(1, activation='sigmoid')(x)
    model = Model(inputs=fe.input, outputs=out,
                  name=f'model_fourier_{n_fourier}')
    model.compile(
        loss='binary_crossentropy',
        optimizer='adam',
        metrics=['accuracy']
    )
    return model

## Utilitários de benchmark

In [ ]:
def model_size_mb(model):
    """Tamanho estimado em MB (parâmetros float32)."""
    n_params = sum(tf.size(w).numpy() for w in model.weights)
    return n_params * 4 / (1024 ** 2)


def count_params(model):
    trainable     = sum(tf.size(w).numpy() for w in model.trainable_weights)
    non_trainable = sum(tf.size(w).numpy() for w in model.non_trainable_weights)
    return trainable, non_trainable


def measure_latency_ms(model, x_sample, n_warmup=5, n_reps=20):
    """Latência mediana de inferência em ms (batch completo)."""
    for _ in range(n_warmup):
        _ = model(x_sample, training=False)
    times = []
    for _ in range(n_reps):
        t0 = time.perf_counter()
        _ = model(x_sample, training=False)
        times.append((time.perf_counter() - t0) * 1000)
    return float(np.median(times))


def load_data(disp, img, folder=FOLDER_I):
    """Carrega splits train/val/test — mesma divisão 60/20/20 do notebook 1."""
    load = lambda fname: pickle.load(open(fname, 'rb'))
    X_tr = load(f"{folder}/X_{img}_train({disp}).pickle")
    y_tr = load(f"{folder}/y_train({disp}).pickle")
    X_va = load(f"{folder}/X_{img}_val({disp}).pickle")
    y_va = load(f"{folder}/y_val({disp}).pickle")
    X_te = load(f"{folder}/X_{img}_test({disp}).pickle")
    y_te = load(f"{folder}/y_test({disp}).pickle")
    return X_tr, y_tr, X_va, y_va, X_te, y_te

## (Opcional) Inspecionar blocos DepthwiseConv2D da MobileNetV3Large

In [ ]:
# Carrega só para inspeção — não treina ainda
_img0 = IMAGENS[0]
_dev0 = DISPOSITIVOS[0]
_X_sample = pickle.load(open(f"{FOLDER_I}/X_{_img0}_train({_dev0}).pickle", 'rb'))
_input_shape = _X_sample.shape[1:]
del _X_sample

_base = MobileNetV3Large(input_shape=_input_shape, weights='imagenet', include_top=False)
dw_info = [
    (i, l.name, l.get_config()['strides'])
    for i, l in enumerate(_base.layers)
    if isinstance(l, DepthwiseConv2D)
]
print(f"Input shape detectado: {_input_shape}")
print(f"Total de blocos DepthwiseConv2D: {len(dw_info)}")
print(f"{'idx':>5}  {'nome':<45}  strides")
for idx, name, strides in dw_info:
    print(f"{idx:>5}  {name:<45}  {strides}")
del _base

## Loop de experimentos

Para cada variante × dispositivo × tipo de imagem:
- Treina `N_RUNS` vezes (mesmo setup do notebook 1)
- Registra acurácia, F1-score, latência e tamanho do modelo

In [ ]:
os.makedirs('output', exist_ok=True)
results = []

for variant_name, n_fourier in VARIANTS.items():
    print(f"\n{'='*65}")
    print(f"VARIANTE: {variant_name}  ({n_fourier} blocos Fourier)")
    print(f"{'='*65}")

    for disp in DISPOSITIVOS:
        for img in IMAGENS:

            X_tr, y_tr, X_va, y_va, X_te, y_te = load_data(disp, img)
            input_shape = X_tr.shape[1:]

            run_accs, run_f1s, latencies = [], [], []
            size_mb, n_train, n_frozen = None, None, None

            for run in range(N_RUNS):
                ckpt = f'ckpt_dft/{variant_name}/{disp}/{img}/run{run}/model.keras'
                os.makedirs(os.path.dirname(ckpt), exist_ok=True)

                model = build_full_model(input_shape, n_fourier)

                # Registra metadados uma vez por combinação
                if run == 0:
                    size_mb  = model_size_mb(model)
                    n_train, n_frozen = count_params(model)

                model.fit(
                    X_tr, y_tr,
                    batch_size=BATCH,
                    epochs=100,
                    verbose=0,
                    validation_data=(X_va, y_va),
                    callbacks=[
                        EarlyStopping(
                            monitor='val_loss', patience=7,
                            restore_best_weights=False, verbose=0
                        ),
                        ModelCheckpoint(
                            ckpt, monitor='val_accuracy',
                            save_best_only=True, verbose=0
                        )
                    ]
                )

                best = tf.keras.models.load_model(
                    ckpt,
                    custom_objects={'SpectralDepthwise': SpectralDepthwise}
                )

                # Métricas
                preds = (best.predict(X_te, batch_size=BATCH, verbose=0) > 0.5).astype(int).flatten()
                run_accs.append(float(np.mean(preds == y_te)))
                run_f1s.append(float(sklearn_f1(y_te, preds, average='macro')))

                # Latência num batch representativo
                x_sample = X_te[:BATCH]
                latencies.append(measure_latency_ms(best, x_sample))

                del model, best
                K.clear_session()

            row = {
                'variant':          variant_name,
                'n_fourier':        n_fourier,
                'device':           disp,
                'image':            img,
                'acc_mean':         np.mean(run_accs),
                'acc_std':          np.std(run_accs),
                'f1_mean':          np.mean(run_f1s),
                'f1_std':           np.std(run_f1s),
                'latency_ms_mean':  np.mean(latencies),
                'latency_ms_std':   np.std(latencies),
                'model_size_mb':    size_mb,
                'trainable_params': n_train,
                'frozen_params':    n_frozen,
            }
            results.append(row)

            print(
                f"  {disp:<16} | {img:<5} | "
                f"acc={np.mean(run_accs):.3f}±{np.std(run_accs):.3f} | "
                f"f1={np.mean(run_f1s):.3f} | "
                f"lat={np.mean(latencies):.1f}ms"
            )

df_results = pd.DataFrame(results)
df_results.to_csv('output/dft_experiment_results.csv', index=False)
print("\nResultados salvos → output/dft_experiment_results.csv")
df_results.head()

## Resumo comparativo entre variantes

In [ ]:
summary = (
    df_results
    .groupby('variant')[['acc_mean', 'f1_mean', 'latency_ms_mean', 'model_size_mb', 'trainable_params']]
    .mean()
    .round(4)
)
print(summary.to_string())

## Visualizações

### Heatmaps de acurácia (dispositivo × tipo de imagem)

In [ ]:
import matplotlib.pyplot as plt

n_variants = len(VARIANTS)
fig, axes = plt.subplots(1, n_variants, figsize=(6 * n_variants, 5))
if n_variants == 1:
    axes = [axes]

for ax, variant_name in zip(axes, VARIANTS):
    sub = df_results[df_results['variant'] == variant_name]
    pivot = sub.pivot(index='device', columns='image', values='acc_mean')
    im = ax.imshow(pivot.values, vmin=0.5, vmax=1.0, cmap='Blues', aspect='auto')
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=45)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_title(f'{variant_name}\n(acurácia média)')
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            val = pivot.values[i, j]
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                    fontsize=8, color='black' if val < 0.8 else 'white')
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.savefig('output/dft_heatmaps_accuracy.png', dpi=150)
plt.show()

### Comparação de latência e tamanho

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

grouped = df_results.groupby('variant')

# Latência
lat = grouped['latency_ms_mean'].mean()
axes[0].bar(lat.index, lat.values, color=['steelblue', 'darkorange', 'green'])
axes[0].set_title('Latência média (ms)')
axes[0].set_ylabel('ms / batch')
axes[0].tick_params(axis='x', rotation=15)

# Acurácia média
acc = grouped['acc_mean'].mean()
axes[1].bar(acc.index, acc.values, color=['steelblue', 'darkorange', 'green'])
axes[1].set_title('Acurácia média')
axes[1].set_ylim(0.5, 1.0)
axes[1].tick_params(axis='x', rotation=15)

# F1 médio
f1 = grouped['f1_mean'].mean()
axes[2].bar(f1.index, f1.values, color=['steelblue', 'darkorange', 'green'])
axes[2].set_title('F1-score médio')
axes[2].set_ylim(0.5, 1.0)
axes[2].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('output/dft_benchmark_summary.png', dpi=150)
plt.show()

print("\nLatência por variante (ms):\n", lat.round(2).to_string())
print("\nAcurácia por variante:\n", acc.round(4).to_string())
print("\nF1-score por variante:\n", f1.round(4).to_string())

### Delta de acurácia: variantes Fourier vs. baseline

In [ ]:
baseline_acc = df_results[df_results['variant'] == 'baseline'].set_index(['device', 'image'])['acc_mean']

for v in [k for k in VARIANTS if k != 'baseline']:
    v_acc = df_results[df_results['variant'] == v].set_index(['device', 'image'])['acc_mean']
    delta = (v_acc - baseline_acc).unstack('image')
    print(f"\n--- Delta acc: {v} − baseline ---")
    print(delta.round(4).to_string())